# 01 — Fact edit + CLT (re)training (Methods 1 & 2)
Runs the MEMIT edit, saves the edited model, computes the apricot target
stats, smoke-tests the training add-on locally, and emits/submits PSC jobs.

In [ ]:
# --- setup ---
import os, sys, socket, json
from pathlib import Path
REPO = Path.cwd()
while REPO.name != "Interp_LM4" and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO))
FACTEDIT = REPO.parent / "FactEditing"
from clts.storage import storage_root
from clts.edit_clt import edit_clt_config as cfg
from clts.edit_clt.prepare_edited_model import make_edited_model
ON_PSC = Path("/jet/home/friedmae").exists()
STORE = storage_root()
print("repo:", REPO, "| on_psc:", ON_PSC, "| storage:", STORE)

In [ ]:
# --- experiment config ---
BASE_CLT = STORE / "clt_runs/grid-L4-H6/standalone/mult16_l02_lr0.0001_ep50_n10000/final"
DATA_DIR = REPO / "data/bioS_N-Bd_final_grid"
C = cfg.default_config(REPO, STORE, base_clt_dir=BASE_CLT, data_dir=DATA_DIR)
EXP_ID = f"{C.edited_model_name()}"
print("edited model:", C.edited_model_name())
for m in C.methods: print(" ", m.key, "->", m.out_tag, "lr", m.lr, "ep", m.epochs)

In [ ]:
# --- baseline target stats: apricot ce_recovered/nmse on the ORIGINAL model ---
import torch
from clts.tl_model import build_hooked_transformer
from clts.clt import CrossLayerTranscoder
from clts.evalCLT import capture_activations, compute_layer_metrics, ce_recovered_full
from clts.export_tokenizer import ensure_hf_tokenizer
from util.condensed_tokenizer import CondensedTokenizer
from util.bio_sampler import BioSampler
from util.diverse_subset import DiverseBioSubset
orig_model = build_hooked_transformer(REPO / "model/grid-L4-H6", "cpu", torch.float32)
base_clt = CrossLayerTranscoder.load_from_dir(BASE_CLT).to("cpu")
tok = CondensedTokenizer.from_remap_path(DATA_DIR / "old_to_new.json")
sub = DiverseBioSubset(BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=1),
                       tok, context_size=512, seed=1)
ev = torch.tensor(sub.to_hf_dataset(64, verbose=False)["input_ids"])
x, y = capture_activations(orig_model, ev)
target_stats = {**compute_layer_metrics(base_clt, x, y),
                **ce_recovered_full(orig_model, base_clt, ev)}
print("apricot target ce_recovered:", round(target_stats["ce_recovered"], 4))

In [ ]:
# --- run the MEMIT edit + save the edited model dir ---
edited_dir = C.edited_model_dir(REPO)
info = make_edited_model(C.fact, edited_dir, factediting_root=FACTEDIT,
                         device="cpu", controls=25)   # raise controls for the real run
print("verified:", info["verified"], "| saved:", info["edited_model_dir"])
import pandas as pd; pd.DataFrame(info["edit_metrics"])

In [ ]:
# --- LOCAL CPU smoke test of the training add-on (tiny budget) ---
# Validates Method 1 (scratch) + Method 2 (resume) code paths end-to-end.
import subprocess
def run(tag, *extra):
    cmd = [sys.executable, "clts/trainCLT.py",
           "--model-dir", str(edited_dir), "--data-dir", str(DATA_DIR),
           "--model-name", C.edited_model_name(),
           "--expansion","16","--l0","2","--n-examples","200","--epochs","1",
           "--eval-every","50", *extra]
    print(">>", tag)
    subprocess.run(cmd, cwd=REPO, check=True,
                   env={**os.environ, "CLT_STORAGE_ROOT": str(STORE), "WANDB_MODE": "disabled"})
run("m1-smoke", "--lr","1e-4","--out-tag","smoke-m1")
run("m2-smoke", "--lr","2e-5","--out-tag","smoke-m2",
    "--resume-from", str(BASE_CLT), "--plateau-patience","2","--plateau-min-delta","0.01")

In [ ]:
# --- emit / submit PSC jobs ---
target_ce = round(target_stats["ce_recovered"], 4)
if ON_PSC:
    env = {**os.environ, "EDITED_MODEL_NAME": C.edited_model_name(),
           "EDITED_MODEL_DIR": str(edited_dir), "DATA_DIR": str(DATA_DIR),
           "BASE_CLT_DIR": str(BASE_CLT), "TARGET_CE_RECOVERED": str(target_ce)}
    subprocess.run(["bash","clts/edit_clt/submit_edit_clt.sh","--test"], cwd=REPO, env=env, check=True)
    print("submitted --test; after it succeeds, rerun without --test for full runs")
else:
    print("Run on PSC:\n  git push  # then on PSC: git pull")
    print(f"  EDITED_MODEL_NAME={C.edited_model_name()} \\\n  "
          f"EDITED_MODEL_DIR=<psc edited dir> DATA_DIR=<psc data> \\\n  "
          f"BASE_CLT_DIR=<psc apricot final> TARGET_CE_RECOVERED={target_ce} \\\n  "
          f"bash clts/edit_clt/submit_edit_clt.sh --test")

In [ ]:
# --- write the manifest (consumed by Notebook 2) ---
methods = {m.key: {"out_tag": m.out_tag, "config": vars(m),
                   "expected_clt_dir": str(cfg.expected_clt_dir(
                       STORE, C.edited_model_name(), m.out_tag,
                       m.expansion, m.l0, m.lr, m.epochs, m.n_examples)),
                   "status": "submitted" if ON_PSC else "pending"}
           for m in C.methods}
first = BioSampler(DATA_DIR / "people.json", fields=("birthday",), seed=0).people[C.fact.person]
trace_prompt = C.trace_prompt_template.format(first=first["first_name"], last=first["last_name"])
manifest = {"exp_id": EXP_ID, "fact": vars(C.fact),
            "orig_model_dir": str(REPO / "model/grid-L4-H6"),
            "edited_model_dir": str(edited_dir), "base_clt_dir": str(BASE_CLT),
            "data_dir": str(DATA_DIR), "target_stats": target_stats,
            "trace_prompt": trace_prompt, "methods": methods}
man_path = STORE / "edit_experiments" / EXP_ID / "manifest.json"
cfg.write_manifest(man_path, manifest)
print("manifest:", man_path)

## Next steps
1. `git push` → on PSC `git pull`; verify apricot + data present.
2. Re-run the edit on PSC (or rsync the edited model up).
3. `bash clts/edit_clt/submit_edit_clt.sh --test` → on success, without `--test`.
4. Sync `clt_runs` back to the Mac, then open `02_circuit_compare.ipynb`.